<a href="https://colab.research.google.com/github/AndrewEisele/Accessibility_App/blob/main/MIST_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Steam Game Accessibility Ranker
-------------------------------
Pick a genre and get the games in that genre ranked by an accessibility
score out of 10 (highest first).

Each game gets four sub-scores (each 0-10) that are averaged into a base score:

  Price      CURRENT price: free = 10; scales down linearly to 0 at PRICE_CAP
  Platform   Windows / Mac / Linux support: 1 of 3 = 3.3, 2 = 6.7, all 3 = 10
  Age        required age 0 = 10; scales down linearly to 0 at AGE_CAP
  Language   number of supported languages, scaled up to LANGUAGE_CAP

Then a "cheapest ever" bonus is added (final score is capped at 10):

  +2  a paid game that is currently ON SALE at its lowest price ever
  +1  a paid game whose current price equals its lowest price ever
      (it has never been cheaper), but is not on sale right now
  +0  anything else (free games get no bonus; they already max the price score)

Lowest-ever prices come from the CheapShark API.

Genre is the filter you choose, so it isn't scored.

Requires: pandas, requests   (pip install pandas requests)

Usage:
    python steam_accessibility.py                  (interactive)
    python steam_accessibility.py my_games.csv     (custom CSV path)
"""

import json
import sys
import time

import pandas as pd
import requests

CSV_PATH = "steam_games.csv"
CACHE_PATH = "cheapshark_cache.json"
CHEAPSHARK_URL = "https://www.cheapshark.com/api/1.0/games"

# Roughly how many top games per genre get a CheapShark lookup. Games close
# enough to the cutoff to be lifted into the top by a bonus are included too.
LOOKUP_LIMIT = 25
MAX_LOOKUPS = 60        # hard ceiling on API calls per genre


# =============================================================================
# Class 1: scoring
# =============================================================================
class AccessibilityScorer:
    """Turns game data into 0-10 accessibility scores."""

    def __init__(self, price_cap=60.0, age_cap=18, language_cap=20,
                 bonus_low=1.0, bonus_sale_low=2.0, weights=None):
        self.price_cap = price_cap              # price at/above this scores 0
        self.age_cap = age_cap                  # age at/above this scores 0
        self.language_cap = language_cap        # languages at/above this scores 10
        self.bonus_low = bonus_low              # at lowest price ever
        self.bonus_sale_low = bonus_sale_low    # on sale AND at lowest price ever
        # Equal weights by default; normalised, so only ratios matter.
        self.weights = weights or {"price": 1, "platform": 1, "age": 1, "language": 1}

    @property
    def max_bonus(self):
        return max(self.bonus_low, self.bonus_sale_low)

    def price_score(self, price):
        """Free = 10, declining linearly to 0 at price_cap."""
        return max(0.0, 10.0 * (1 - price / self.price_cap))

    def platform_score(self, row):
        """Share of the three platforms supported, out of 10."""
        return 10.0 * (int(row["windows"]) + int(row["mac"]) + int(row["linux"])) / 3

    def age_score(self, age):
        """Age 0 = 10, declining linearly to 0 at age_cap."""
        return max(0.0, 10.0 * (1 - age / self.age_cap))

    def language_score(self, n_languages):
        """More languages = higher score, maxing out at language_cap."""
        return min(10.0, 10.0 * n_languages / self.language_cap)

    def low_price_bonus(self, row):
        """Bonus for a paid game currently at its lowest price ever.
        Needs a 'lowest_price' value (NaN/None = unknown = no bonus)."""
        lowest = row.get("lowest_price")
        if row["price"] <= 0 or lowest is None or pd.isna(lowest):
            return 0.0
        if row["price"] <= lowest + 0.005:          # at (or below) its all-time low
            return self.bonus_sale_low if row["discount"] > 0 else self.bonus_low
        return 0.0

    def score(self, games):
        """Return a copy of `games` with sub-scores, bonus and final score,
        sorted highest first."""
        games = games.copy()
        if "lowest_price" not in games.columns:
            games["lowest_price"] = float("nan")

        games["price_score"] = games["price"].apply(self.price_score)
        games["platform_score"] = games.apply(self.platform_score, axis=1)
        games["age_score"] = games["required_age"].apply(self.age_score)
        games["language_score"] = games["n_languages"].apply(self.language_score)

        total = sum(self.weights.values())
        games["base_score"] = (
            games["price_score"] * self.weights["price"]
            + games["platform_score"] * self.weights["platform"]
            + games["age_score"] * self.weights["age"]
            + games["language_score"] * self.weights["language"]
        ) / total

        games["low_bonus"] = games.apply(self.low_price_bonus, axis=1)
        games["accessibility_score"] = (games["base_score"] + games["low_bonus"]).clip(upper=10.0)

        # Highest first; ties broken by cheaper price, then name
        return games.sort_values(
            ["accessibility_score", "price", "name"], ascending=[False, True, True]
        ).reset_index(drop=True)


# =============================================================================
# Class 2: CheapShark web API client
# =============================================================================
class CheapSharkClient:
    """Looks up the lowest price a game has ever had, using the free
    CheapShark API (https://www.cheapshark.com/api). Results are cached in a
    JSON file so repeat lookups are instant and the API isn't hammered."""

    def __init__(self, cache_path=CACHE_PATH, delay=0.25, timeout=10):
        self.cache_path = cache_path
        self.delay = delay          # pause between calls to be polite
        self.timeout = timeout
        self.available = True       # flips to False if the API can't be reached
        self.cache = self._load_cache()

    # ---- cache ----
    def _load_cache(self):
        try:
            with open(self.cache_path, encoding="utf-8") as f:
                return json.load(f)
        except (OSError, ValueError):
            return {}

    def _save_cache(self):
        try:
            with open(self.cache_path, "w", encoding="utf-8") as f:
                json.dump(self.cache, f)
        except OSError:
            pass

    # ---- API calls ----
    def _get(self, params):
        response = requests.get(CHEAPSHARK_URL, params=params, timeout=self.timeout)
        response.raise_for_status()
        time.sleep(self.delay)
        return response.json()

    def _game_id(self, steam_app_id):
        """Steam appid -> CheapShark gameID (or None if not tracked)."""
        results = self._get({"steamAppID": int(steam_app_id)})
        if results:
            return results[0].get("gameID")
        return None

    def lowest_prices(self, steam_app_ids, progress=True):
        """Return {steam_app_id: lowest_price_ever} for the ids CheapShark
        knows about. Unknown games are simply left out."""
        wanted = [str(int(a)) for a in steam_app_ids]
        todo = [a for a in wanted if a not in self.cache]

        if todo and self.available:
            try:
                # 1) Steam appid -> CheapShark gameID (one call per game)
                id_map = {}
                for n, app_id in enumerate(todo, 1):
                    if progress:
                        print(f"\r  Checking lowest prices on CheapShark... {n}/{len(todo)}",
                              end="", flush=True)
                    game_id = self._game_id(app_id)
                    if game_id:
                        id_map[game_id] = app_id
                    else:
                        self.cache[app_id] = None   # not tracked; remember that

                # 2) gameIDs -> price history, 25 per call
                game_ids = list(id_map)
                for i in range(0, len(game_ids), 25):
                    batch = game_ids[i:i + 25]
                    data = self._get({"ids": ",".join(batch)})
                    for game_id, info in data.items():
                        try:
                            price = float(info["cheapestPriceEver"]["price"])
                        except (KeyError, TypeError, ValueError):
                            price = None
                        self.cache[id_map[game_id]] = price
                if progress:
                    print()
            except (requests.RequestException, ValueError) as err:
                if progress:
                    print()
                print(f"  (CheapShark unavailable: {err.__class__.__name__}. "
                      "No 'cheapest ever' bonuses will be applied.)")
                self.available = False
            finally:
                self._save_cache()

        return {int(a): self.cache[a] for a in wanted if self.cache.get(a) is not None}


# =============================================================================
# Loading and genre helpers
# =============================================================================
def load_games(path):
    df = pd.read_csv(path)

    df["price"] = pd.to_numeric(df["price"], errors="coerce").fillna(0.0)
    df["discount"] = pd.to_numeric(df["discount"], errors="coerce").fillna(0)
    df["required_age"] = pd.to_numeric(df["required_age"], errors="coerce").fillna(0)
    df["n_languages"] = pd.to_numeric(df["n_languages"], errors="coerce").fillna(0)
    for col in ("windows", "mac", "linux"):
        df[col] = df[col].astype(str).str.lower().eq("true")

    # Genres are stored as a comma-separated string, e.g. "Action, Free To Play"
    df["genre_list"] = df["genres"].fillna("").apply(
        lambda s: [g.strip() for g in s.split(",") if g.strip()]
    )
    return df


def genre_counts(df):
    return df.explode("genre_list")["genre_list"].dropna().value_counts()


def resolve_genre(user_input, counts):
    """Match input to a genre, case-insensitively, allowing partial matches."""
    text = user_input.strip().lower()
    for genre in counts.index:
        if genre.lower() == text:
            return genre
    matches = [g for g in counts.index if text and text in g.lower()]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        print(f"  '{user_input}' matches several genres: {', '.join(matches)}")
    else:
        print(f"  No genre found matching '{user_input}'.")
    return None


# =============================================================================
# Ranking
# =============================================================================
def rank_genre(df, genre, scorer, client, lookup_limit=LOOKUP_LIMIT):
    """Score every game in the genre using its CURRENT price, then check
    CheapShark for 'cheapest ever' bonuses on the games that could end up near
    the top (paid games within one bonus of the top `lookup_limit`)."""
    subset = df[df["genre_list"].apply(lambda gl: genre in gl)].copy()

    # Pass 1: rank everything with no bonuses
    ranked = scorer.score(subset)
    if lookup_limit <= 0:
        return ranked

    # Games that could reach the top `lookup_limit` once a bonus is added
    cutoff = ranked["base_score"].iloc[min(lookup_limit, len(ranked)) - 1] - scorer.max_bonus
    candidates = ranked[(ranked["base_score"] >= cutoff) & (ranked["price"] > 0)]
    candidates = candidates.head(MAX_LOOKUPS)

    # Pass 2: fetch lowest-ever prices, then re-score with bonuses
    lowest = client.lowest_prices(candidates["appid"])
    if lowest:
        ranked["lowest_price"] = ranked["appid"].map(lowest)
        ranked = scorer.score(ranked)
    return ranked


def show_results(ranked, genre, used_cheapshark=True):
    print(f"\nAccessibility ranking for '{genre}' ({len(ranked)} games)")
    print("Score is out of 10 = average of Price / Platform / Age / Language, plus a")
    print("bonus (+1 or +2, max 10) if the game is at its lowest price ever (+2 if on sale).")
    if not used_cheapshark:
        print("Note: CheapShark was unavailable, so no bonuses were applied.")
    print()

    header = (f"{'#':>3}  {'Game':<38} {'Score':>5}  {'Price':>7} {'Low':>7} {'Bonus':>5}  "
              f"{'Pri':>4} {'Plat':>4} {'Age':>4} {'Lang':>4}")
    print(header)
    print("-" * len(header))
    for i, row in ranked.iterrows():
        name = row["name"] if len(row["name"]) <= 38 else row["name"][:35] + "..."
        price = "Free" if row["price"] == 0 else f"${row['price']:.2f}"
        low = "-" if pd.isna(row["lowest_price"]) else (
            "Free" if row["lowest_price"] == 0 else f"${row['lowest_price']:.2f}")
        bonus = f"+{row['low_bonus']:.0f}" if row["low_bonus"] else ""
        print(
            f"{i + 1:>3}  {name:<38} {row['accessibility_score']:>5.1f}  {price:>7} {low:>7} {bonus:>5}  "
            f"{row['price_score']:>4.1f} {row['platform_score']:>4.1f} "
            f"{row['age_score']:>4.1f} {row['language_score']:>4.1f}"
        )


def export_csv(ranked, genre):
    safe = "".join(c if c.isalnum() else "_" for c in genre).strip("_").lower()
    filename = f"accessibility_{safe}.csv"
    cols = [
        "name", "accessibility_score", "base_score", "low_bonus", "price_score",
        "platform_score", "age_score", "language_score", "price", "lowest_price",
        "discount", "required_age", "windows", "mac", "linux", "n_languages", "genres",
    ]
    ranked[cols].round(2).to_csv(filename, index=False)
    print(f"\nSaved to {filename}")


# =============================================================================
# Main loop
# =============================================================================
def main():
    # Check if running in Google Colab or IPython interactive environment
    is_colab = "google.colab" in sys.modules or "IPython" in sys.modules

    # Ignore command line arguments if running interactively in Colab/IPython
    if is_colab:
        path = CSV_PATH
    else:
        path = sys.argv[1] if len(sys.argv) > 1 else CSV_PATH

    try:
        df = load_games(path)
    except FileNotFoundError:
        print(f"Could not find '{path}'. Put it next to this script or pass its path.")
        return

    scorer = AccessibilityScorer()
    client = CheapSharkClient()
    counts = genre_counts(df)
    print(f"Loaded {len(df)} games across {len(counts)} genres.")

    while True:
        print("\nAvailable genres:")
        print(", ".join(f"{g} ({n})" for g, n in counts.items()))
        choice = input("\nEnter a genre (or 'q' to quit): ").strip()
        if choice.lower() in ("q", "quit", "exit"):
            print("Goodbye!")
            break

        genre = resolve_genre(choice, counts)
        if genre is None:
            continue

        ranked = rank_genre(df, genre, scorer, client)
        show_results(ranked, genre, used_cheapshark=client.available)
        if input("\nSave this ranking to a CSV? (y/n): ").strip().lower() == "y":
            export_csv(ranked, genre)


if __name__ == "__main__":
    main()


Loaded 1000 games across 12 genres.

Available genres:
Action (627), Indie (429), Adventure (425), RPG (269), Simulation (256), Strategy (205), Free To Play (189), Casual (146), Massively Multiplayer (108), Early Access (69), Sports (39), Racing (31)

Enter a genre (or 'q' to quit): Action
  Checking lowest prices on CheapShark... 1/60
  (CheapShark unavailable: HTTPError. No 'cheapest ever' bonuses will be applied.)

Accessibility ranking for 'Action' (627 games)
Score is out of 10 = average of Price / Platform / Age / Language, plus a
bonus (+1 or +2, max 10) if the game is at its lowest price ever (+2 if on sale).
Note: CheapShark was unavailable, so no bonuses were applied.

  #  Game                                   Score    Price     Low Bonus   Pri Plat  Age Lang
---------------------------------------------------------------------------------------------
  1  Combat Master: Season 3                 10.0     Free       -        10.0 10.0 10.0 10.0
  2  DDNet                    

KeyboardInterrupt: Interrupted by user